# DeepEval: Production Evaluations with DeepEval

DeepEval is the RECOMMENDED framework for evaluating LLM outputs in production.
Think of it as "pytest for LLMs." It provides pre-built, battle-tested metrics
so you don't have to roll your own judge from scratch.

**Why DeepEval over rolling your own LLM-as-Judge?**
- Pre-built metrics that have been tested and validated
- Consistent scoring methodology across your whole team
- Pytest integration -- run evals in CI/CD like any other test
- Dashboard (Confident AI) for tracking results over time
- Active open-source community

**Note:** DeepEval uses OpenAI by default for its evaluation models.
Set `OPENAI_API_KEY` if using the default evaluation model.

**Documentation:** https://docs.confident-ai.com

## Install Dependencies

In [ ]:
!pip install deepeval

## Imports

These are the core building blocks of DeepEval:
- `LLMTestCase`: A single test case (input, output, context)
- `evaluate`: Run metrics against test cases
- **Metrics**: Specific evaluation criteria (hallucination, relevancy, etc.)

In [ ]:
import os

from deepeval import evaluate
from deepeval.test_case import LLMTestCase
from deepeval.metrics import (
    HallucinationMetric,
    AnswerRelevancyMetric,
    FaithfulnessMetric,
    GEval,
)

## Example 1: Basic Hallucination Check

The Hallucination Metric checks whether the LLM's output contains
information that is NOT supported by the provided context.

This is critical for RAG systems where the LLM should ONLY use
information from retrieved documents.

A hallucination is when the LLM says something that is NOT in the
provided context. For example, if the context says "median price is
$750,000" but the LLM says "$850,000", that's a hallucination.

In [ ]:
def example_hallucination_check():
    """
    Check if an LLM response contains hallucinated information.

    A hallucination is when the LLM says something that is NOT in the
    provided context. For example, if the context says "median price is
    $750,000" but the LLM says "$850,000", that's a hallucination.
    """

    print("=" * 60)
    print("EXAMPLE 1: Hallucination Detection")
    print("=" * 60)

    # Create a test case with:
    #   - input: what the user asked
    #   - actual_output: what the LLM actually said
    #   - context: the facts the LLM was supposed to use
    test_case = LLMTestCase(
        input="What is the median home price in Kailua, Hawaii?",
        actual_output=(
            "The median home price in Kailua, Hawaii is approximately $1,250,000 "
            "as of Q4 2025. The market has seen a 3.8% year-over-year increase. "
            "Homes in Kailua typically spend about 28 days on the market."
        ),
        context=[
            "The median home price in Kailua, Hawaii is $1,250,000 as of Q4 2025.",
            "Year-over-year price change in Kailua is +3.8%.",
            "Average days on market in Kailua is 28 days.",
            "There are currently 45 active listings in Kailua."
        ]
    )

    # Create the hallucination metric
    # threshold: minimum score to PASS (0.0 to 1.0)
    # A higher threshold means stricter evaluation
    hallucination_metric = HallucinationMetric(
        threshold=0.5,  # Must score above 0.5 to pass
    )

    # Evaluate
    hallucination_metric.measure(test_case)

    print(f"Score: {hallucination_metric.score}")
    print(f"Passed: {hallucination_metric.is_successful()}")
    print(f"Reason: {hallucination_metric.reason}")
    print()

    return test_case, hallucination_metric

In [ ]:
example_hallucination_check()

## Example 2: Answer Relevancy Check

The Answer Relevancy Metric checks whether the LLM's response is
actually relevant to the question that was asked.

An irrelevant response might be factually correct but doesn't answer
the user's actual question.

In [ ]:
def example_relevancy_check():
    """
    Check if an LLM response is relevant to the question asked.

    A response can be factually correct but still fail relevancy if it
    doesn't address what the user actually asked.
    """

    print("=" * 60)
    print("EXAMPLE 2: Answer Relevancy")
    print("=" * 60)

    # Good response — directly answers the question
    good_test_case = LLMTestCase(
        input="What should I budget for closing costs when buying a home in Hawaii?",
        actual_output=(
            "When buying a home in Hawaii, you should budget approximately 2-5% of "
            "the purchase price for closing costs. This includes title insurance, "
            "escrow fees, recording fees, and prorated property taxes. For a "
            "$1,000,000 home, expect $20,000 to $50,000 in closing costs. Hawaii "
            "also has a conveyance tax that varies based on the sale price."
        ),
    )

    # Bad response — talks about something else entirely
    bad_test_case = LLMTestCase(
        input="What should I budget for closing costs when buying a home in Hawaii?",
        actual_output=(
            "Hawaii is a beautiful state with wonderful beaches and a tropical climate. "
            "The islands offer diverse landscapes from volcanic mountains to lush "
            "rainforests. Many people dream of living in Hawaii for its outdoor "
            "lifestyle and aloha spirit."
        ),
    )

    relevancy_metric = AnswerRelevancyMetric(
        threshold=0.7,  # Must score above 0.7 to pass
    )

    # Evaluate the good response
    print("--- Good Response ---")
    relevancy_metric.measure(good_test_case)
    print(f"Score: {relevancy_metric.score}")
    print(f"Passed: {relevancy_metric.is_successful()}")
    print(f"Reason: {relevancy_metric.reason}")
    print()

    # Evaluate the bad response
    print("--- Bad Response ---")
    relevancy_metric.measure(bad_test_case)
    print(f"Score: {relevancy_metric.score}")
    print(f"Passed: {relevancy_metric.is_successful()}")
    print(f"Reason: {relevancy_metric.reason}")
    print()

    return good_test_case, bad_test_case, relevancy_metric

In [ ]:
example_relevancy_check()

## Example 3: Faithfulness Check

Faithfulness measures whether every claim in the LLM's output can be
traced back to the provided context. This is closely related to
hallucination detection but focuses on "is every claim supported?"

In [ ]:
def example_faithfulness_check():
    """
    Check if every claim in the response is faithful to the source context.
    """

    print("=" * 60)
    print("EXAMPLE 3: Faithfulness")
    print("=" * 60)

    test_case = LLMTestCase(
        input="Summarize the Kailua real estate market.",
        actual_output=(
            "The Kailua real estate market is strong, with a median home price of "
            "$1,250,000 and only 1.8 months of inventory, indicating a seller's "
            "market. Homes are selling in an average of 28 days, and prices have "
            "increased 3.8% year over year."
        ),
        retrieval_context=[
            "Kailua median sale price: $1,250,000 (up 3.8% YoY).",
            "Average days on market in Kailua: 28.",
            "Months of inventory: 1.8 (seller's market).",
            "Number of sales this quarter: 87.",
            "Price per sq ft: $785."
        ]
    )

    faithfulness_metric = FaithfulnessMetric(
        threshold=0.7,
    )

    faithfulness_metric.measure(test_case)

    print(f"Score: {faithfulness_metric.score}")
    print(f"Passed: {faithfulness_metric.is_successful()}")
    print(f"Reason: {faithfulness_metric.reason}")
    print()

    return test_case, faithfulness_metric

In [ ]:
example_faithfulness_check()

## Example 4: Custom Evaluation with G-Eval

G-Eval lets you define CUSTOM evaluation criteria using natural language.
This is like creating your own LLM-as-Judge but within DeepEval's
framework, so you get all the benefits (tracking, pytest, etc.).

This is useful when the built-in metrics don't cover your specific
use case.

In [ ]:
def example_custom_geval():
    """
    Create a custom evaluation metric using G-Eval.

    G-Eval uses an LLM to evaluate based on criteria YOU define.
    This is useful when the built-in metrics don't cover your specific
    use case.
    """

    print("=" * 60)
    print("EXAMPLE 4: Custom G-Eval (Professionalism)")
    print("=" * 60)

    # Define a custom "professionalism" metric for real estate communications
    professionalism_metric = GEval(
        name="Professionalism",
        criteria=(
            "Evaluate whether the response is written in a professional tone "
            "appropriate for a real estate agent communicating with clients. "
            "It should be warm but professional, avoid slang, use proper "
            "grammar, and convey expertise and trustworthiness."
        ),
        evaluation_params=[
            "input",
            "actual_output"
        ],
        threshold=0.7,
    )

    # Test with a professional response
    professional_case = LLMTestCase(
        input="Write a follow-up email to a buyer who just viewed a home.",
        actual_output=(
            "Subject: Thank You for Visiting 123 Kailua Road\n\n"
            "Dear Sarah,\n\n"
            "Thank you for taking the time to tour the home at 123 Kailua Road "
            "yesterday. I hope you enjoyed seeing the renovated kitchen and the "
            "beautiful ocean views from the lanai.\n\n"
            "I'd love to hear your thoughts on the property. If you have any "
            "questions about the home, the neighborhood, or the buying process, "
            "please don't hesitate to reach out.\n\n"
            "I also have two other properties in Kailua that might interest you "
            "based on your preferences. Would you like me to schedule viewings?\n\n"
            "Best regards,\n"
            "Charlie"
        ),
    )

    # Test with an unprofessional response
    unprofessional_case = LLMTestCase(
        input="Write a follow-up email to a buyer who just viewed a home.",
        actual_output=(
            "yo sarah!! that house was sick right??\n\n"
            "lmk if u wanna buy it lol. its gonna go fast tho so u better "
            "hurry up and decide. no pressure haha but seriously tho.\n\n"
            "hit me up whenever\n"
            "charlie"
        ),
    )

    print("--- Professional Response ---")
    professionalism_metric.measure(professional_case)
    print(f"Score: {professionalism_metric.score}")
    print(f"Passed: {professionalism_metric.is_successful()}")
    print(f"Reason: {professionalism_metric.reason}")
    print()

    print("--- Unprofessional Response ---")
    professionalism_metric.measure(unprofessional_case)
    print(f"Score: {professionalism_metric.score}")
    print(f"Passed: {professionalism_metric.is_successful()}")
    print(f"Reason: {professionalism_metric.reason}")
    print()

    return professionalism_metric

In [ ]:
example_custom_geval()

## Example 5: Running Multiple Metrics at Once (Batch Evaluation)

In production, you'll want to run MULTIPLE metrics on the same test case.
DeepEval's `evaluate()` function makes this easy.

This is the pattern you'd use in production:
1. Define your test cases
2. Define your metrics
3. Run `evaluate()` to score everything

In [ ]:
def example_batch_evaluation():
    """
    Run multiple evaluation metrics on a set of test cases at once.

    This is the pattern you'd use in production:
        1. Define your test cases
        2. Define your metrics
        3. Run evaluate() to score everything
    """

    print("=" * 60)
    print("EXAMPLE 5: Batch Evaluation (Multiple Metrics)")
    print("=" * 60)

    # Define test cases — in production, these would come from your
    # application's actual inputs and outputs
    test_cases = [
        LLMTestCase(
            input="What's the inventory situation in Kailua?",
            actual_output=(
                "Kailua currently has 1.8 months of inventory with 45 active "
                "listings. This indicates a strong seller's market, as a balanced "
                "market typically has 4-6 months of inventory. Homes are selling "
                "in an average of 28 days."
            ),
            context=[
                "Kailua months of inventory: 1.8",
                "Active listings in Kailua: 45",
                "Average days on market: 28",
                "A balanced market has 4-6 months of inventory."
            ],
            retrieval_context=[
                "Kailua months of inventory: 1.8",
                "Active listings in Kailua: 45",
                "Average days on market: 28",
            ]
        ),
        LLMTestCase(
            input="Has the Kailua market gone up or down?",
            actual_output=(
                "The Kailua real estate market has been trending upward. The median "
                "home price increased 3.8% year over year, reaching $1,250,000. "
                "Additionally, homes are selling faster than last year, with average "
                "days on market decreasing from 35 to 28 days."
            ),
            context=[
                "Kailua median sale price: $1,250,000",
                "Year-over-year price change: +3.8%",
                "Average days on market: 28 (down from 35 last year)",
            ],
            retrieval_context=[
                "Kailua median sale price: $1,250,000",
                "Year-over-year price change: +3.8%",
                "Average days on market: 28 (down from 35 last year)",
            ]
        ),
    ]

    # Define metrics to run
    metrics = [
        HallucinationMetric(threshold=0.5),
        AnswerRelevancyMetric(threshold=0.7),
        FaithfulnessMetric(threshold=0.7),
    ]

    # Run all metrics on all test cases
    # This is the one-liner that makes DeepEval powerful
    results = evaluate(
        test_cases=test_cases,
        metrics=metrics,
    )

    print(f"\nBatch evaluation complete.")
    print(f"Results: {results}")
    print()

    return results

In [ ]:
example_batch_evaluation()

## Next Steps for Production

1. **Run evals with pytest:**
   ```
   deepeval test run deepeval_example.py
   ```

2. **Track results over time with Confident AI:**
   ```
   deepeval login
   deepeval test run deepeval_example.py --confident
   ```

3. **Add evals to your CI/CD pipeline:**
   ```yaml
   # In your GitHub Actions / CI config:
   - run: deepeval test run tests/test_evals.py
   ```

4. **Create custom metrics for your domain:**
   Use GEval (shown in Example 4) to define metrics specific to your use case.

5. **Build a test dataset:**
   Collect real inputs/outputs from your application and use them as test cases.
   The more representative your test data, the more useful your evals.

For more info: https://docs.confident-ai.com